# Parmegiani: *Géologie sonore*

## Introduction

In this notebook, we examine the beginning of Parmegiani's *Géologie sonore* from *De Natura Sonorum*, 1re Série.
To quote Parmegiani himself: "Géologie sonore" (Sound Geology) is similar to a flight over an area where different "sound" layers come to the surface one after another.

We focus in particular on the soundscape used as the first "layer" at the beginning of the piece. As referenced by Smalley: "The movement starts with a grounding in a
bass drone, rooted by the interval of a fifth."

## Goals

Goals for this analysis:
- How would you describe the characteristics of the sound at the beginning?
- What is responsible for the "warm," "analog" sound?
- Can we find the fifth Smalley is referring to?



## Outline

- Load and trim the source sample
- Inspect the spectrogram
- Analyze the static spectrum as a single DFT frame
- Refine peak detection and inspect partials
- Resynthesize and compare with the original
- Test the role of the fifth and search for additional `f0` values
- Reconstruct the spectrum from multiple fundamentals
- Visualize the result as symbolic notation (MEI)

## References

https://maison-ona.com/catalog-0160ONA

Smalley D. Space-form and the acousmatic image. Organised Sound. 2007;12(1):35-58. doi:10.1017/S1355771807001665

## Load and Trim the Sample

Let's load and trim the sample:

In [ ]:
from audiospylt.audio_utils import load_audio_sample_and_preview

audio_opts = dict(
    wav_source='../samples/de_natura_son.wav',
    desired_sample_rate=None,
    convert_to_mono=True,
    download_mode='temp',
    download_dir=None,
    show_load_bar=True,
    show_properties=True,
    show_waveform=True,
    play_audio=True,
    playback_fs_target_name='source',
    playback_bit_rate=24,
    playback_save_audio=False,
    playback_sanitize=True,
    source_verbose=True,
    playback_verbose=False,
    plot_width=1200,
    plot_height=600,
    plot_title_source='source',
    plot_title_mode='full',
)

audio_data, sample_rate, audio_info = load_audio_sample_and_preview(**audio_opts)

In [ ]:
from audiospylt.audio_utils import trim_and_fade_and_render

start_time = 1
end_time = 6
add_fades = True
fade_in_duration = 0.05
fade_out_duration = 0.05
fade_in_exponent = 1
fade_out_exponent = 1

cut_opts = dict(
    start_time=start_time,
    end_time=end_time,
    add_fades=add_fades,
    fade_in_duration=fade_in_duration,
    fade_out_duration=fade_out_duration,
    fade_in_exponent=fade_in_exponent,
    fade_out_exponent=fade_out_exponent,
    plot_width=1200,
    plot_height=None,
    plot_title_source='source',
    plot_title_mode='full',
    cut_sanitize=True,
    cut_verbose=True,
    render=True,
    fs_target_name='source',
    bit_rate=24,
    save_audio=False,
    player=True,
    sanitize=True,
    verbose=True,
)

cut_audio_data, cut_duration, cut_render_result = trim_and_fade_and_render(
    audio_data=audio_data,
    sample_rate=sample_rate,
    audio_info=audio_info,
    **cut_opts,
)


## Spectrogram Overview

Let's make a spectrogram and analyze the static nature of the sound. Can we see the fifth Smalley mentioned?

In [ ]:
from audiospylt.audio_utils import plot_spectrogram_with_waveform

# Set options here, then make a single call that applies them.
#
# Parameters (spec_opts):
# - y_axis_mode: 'linear' | 'log' | 'mel' | 'mixed'
# - y_axis_mix: 0..1 (only for 'mixed')
# - mixed_log_floor_hz: >0 (only for 'mixed')
# - amp_scale: 'db' (log) | 'linear' (raw) | 'linear_norm' (0-1)
# - power_law_gamma: <1.0 (e.g. 0.5) to compress the range for linear modes
# - n_fft: FFT window length in samples
# - window_type: scipy window name (e.g., 'hann'); full list: https://docs.scipy.org/doc/scipy/reference/signal.windows.html
# - overlap: 0..(<0.95) fraction of window overlap
# - oversample_factor: >=1.0 zero-padding multiplier (nfft = n_fft * factor)
# - boundary:
#   - 'zeros' (centered): pads both sides; the first frame includes pre-zero padding
#   - 'zeros_end' (start-aligned): pads only the end; the first full window starts at t=0
# - padded: if True, pads the end so the last frame reaches the end
# - time_reference:
#   - 'center': x is window centers (scipy default)
#   - 'start': x is window start times (removes empty area at the beginning for 'zeros_end')
# - time_range: 'signal' shows 0..duration; 'stft' shows only the returned STFT span
# - waveform_mode:
#   - 'integrated': waveform subplot under the spectrogram (single figure)
#   - 'separate': waveform shown as a separate figure
#   - 'none': spectrogram only
# - waveform_height_ratio: height share for waveform when integrated
# - waveform_line_width: waveform line width
# - waveform_max_points: downsample waveform for speed (int max points), or 'sample_rate' to use sample_rate
# - mel_bins/mel_fmax: only used when y_axis_mode='mel'
# - scaling: 'density' | 'spectrum'
# - mode: 'magnitude' | 'psd'
# - cmap: Plotly colorscale name
# - show: display the figure(s)
# - print_info: print derived STFT settings under the plot (hop, overlap %, redundancy, etc.)
# - plot_width/plot_height: optional figure size (pixels)
# - plot_title_source: which string to show in the title: 'filename' | 'source' (uses audio_info)
# - plot_title_mode: how to show it: 'full' | 'basename'

spec_opts = dict(
    audio_info=audio_info,

    # Axis scaling:
    y_axis_mode="log",
    y_axis_mix=0.5,
    mixed_log_floor_hz=1.0,
    amp_scale="db",
    power_law_gamma=0.1,

    # STFT params:
    n_fft=8192,
    window_type="hann",
    overlap=0.75,
    oversample_factor=2.0,

    # Edge framing / time base:
    boundary="zeros_end",
    padded=True,
    time_reference="start",
    time_range="signal",

    # Optional synchronous waveform:
    waveform_mode="integrated",
    waveform_height_ratio=0.25,
    waveform_line_width=1.0,
    waveform_max_points="sample_rate",

    # Mel-only params:
    mel_bins=128,
    mel_fmax=sample_rate / 2,

    # Scaling/appearance:
    scaling="density",
    mode="magnitude",
    cmap="Magma",
    show=True,
    print_info=True,

    # Plot size / title naming:
    plot_width=1200,
    plot_height=None,
    plot_title_source="source",
    plot_title_mode="full",
)

fig, spec_info = plot_spectrogram_with_waveform(cut_audio_data, sample_rate, **spec_opts)



## Single-Frame Spectrum Analysis

Although we clearly see an overtone-like structure, let's use audiospylt's spectrum representation. Because the sound is very static, we can use one window across the full duration and handle the analysis as a single DFT frame. Since we added fade-ins and fade-outs, we do not need a dedicated window function here (just select `boxcar` in the window settings).

`analyze_signal(...)` is optimized for peak detection and threshold-based selection, so we define the threshold window as:

```python
window_type = "boxcar"
thresh_amp_low = 0.0005
thresh_amp_high = 0.05
thresh_freq_low = 20
thresh_freq_high = 1500
```

We also keep peak-detection sensitivity very low:

```python
prominence_rel = 0.0001
width_hz = 0.1
distance_hz = 0.1
```

In [ ]:
from audiospylt.dft_analysis import analyze_signal

# Peak-picking parameters (keep within sensible bounds):
# - window_type: scipy window name
# - thresh_amp_low/high: peak height bounds in *linear FFT magnitude* units (0 < low < high)
# - thresh_freq_low/high: keep peaks within [low, high] Hz (high <= Nyquist or None)
# - prominence: (advanced) raw scipy prominence in the same units as the spectrum amplitude
# - prominence_rel: (recommended) [0,1] prominence as a fraction of max(spec)
# - width: (advanced) raw scipy width in FFT bins
# - width_hz: (recommended) width in Hz (converted to bins)
# - distance_hz: (optional) minimum spacing between peaks in Hz (helps avoid picking adjacent bins)
# - freq_axis_mode: 'linear' | 'log' | 'mel' | 'mixed' (plot only)
# - freq_axis_mix: [0,1] only for 'mixed' (0 = linear spacing, 1 = log-like spacing)
# - mixed_log_floor_hz: >0 only for 'mixed' (plot only)
# - amp_axis_mode: 'linear' | 'log' | 'mixed' (plot only)
# - amp_axis_mix: [0,1] only for amp_axis_mode='mixed'
# - amp_log_floor: >0 floor used for log/mixed amplitude plotting (avoids log(0))
# - auto_plot_range: auto-zoom plot ranges based on freq/amp thresholds (+ padding)
# - freq_plot_pad_hz/freq_plot_pad_frac: padding around [thresh_freq_low, thresh_freq_high] (Hz or fraction of span)
# - amp_plot_pad/amp_plot_pad_frac: additive padding around [thresh_amp_low, thresh_amp_high] (linear/mixed)
# - amp_plot_pad_ratio: multiplicative padding for log amp axis
# - plot_width/plot_height: plot size (pixels) for waveform+window preview and spectrum plot
# - plot_title_source: which string to show in plot titles: 'filename' | 'source'
# - plot_title_mode: how to show it: 'full' | 'basename'
# - show_windowed_waveform: if True, show the waveform after applying window_type, with the window curve overlaid in red (shown first)
# - partial_tracking: if True, keep only peaks within +/- partial_bandwidth_hz of k*f0 (applies after initial peak filtering)
# - f0: fundamental frequency in Hz (required when partial_tracking=True)
# - partial_bandwidth_hz: bandwidth in Hz around each partial (k*f0)
# - plot_partials: if True, overlay partial center lines and +/- bandwidth bands on the spectrum
#
# Highlight / styling parameters (Plotly color strings):
# - spectrum_color: None (default) or a color (e.g., 'royalblue', '#1f77b4', 'rgba(31,119,180,1)')
# - peaks_color: None (default) or a color string
# - threshold_color: color for threshold lines (default: 'Red')
# - partials_color: color for partial center lines (default: 'rgba(0, 180, 0, 0.65)')
# - partials_band_fill: fill color for partial bands (default: 'rgba(0, 180, 0, 0.10)')
#
# Notes:
# - thresh_amp_* values are in *linear FFT magnitude* units (not dB).
# - For amp_axis_mode='mixed', y-axis tick labels are mapped back to raw amplitude units.

window_type = "boxcar"
thresh_amp_low = 0.0005
thresh_amp_high = 0.05 
thresh_freq_low = 20 
thresh_freq_high = 1500 

# Prefer the intuitive variants:
prominence_rel = 0.0001 # 0-1 range, try smaller values for more sensitive detection
width_hz = 0.1        # e.g. 5.0..30.0 to prefer broader peaks
distance_hz = 0.1     # e.g. 5.0..30.0 to avoid adjacent-bin peaks

# Keep these as None unless you want raw scipy behavior:
prominence = None
width = None

# Plot axis scaling:
freq_axis_mode = "linear"
freq_axis_mix = 0.5
mixed_log_floor_hz = 1.0

amp_axis_mode = "linear"
amp_axis_mix = 0.5
amp_log_floor = 1e-12

# Auto-zoom plot ranges to your thresholds (+ padding):
auto_plot_range = True
freq_plot_pad_hz = None      # set e.g. 50 for +/-50 Hz padding
freq_plot_pad_frac = 0.05    # used when freq_plot_pad_hz is None
amp_plot_pad = None          # set e.g. 0.01 for +/-0.01 amplitude padding
amp_plot_pad_frac = 0.10     # used when amp_plot_pad is None
amp_plot_pad_ratio = 0.15    # only used when amp_axis_mode='log'

# Plot size / naming:
plot_width = 1200
plot_height = None
plot_title_source = "source"   # "filename" | "source"
plot_title_mode = "basename"     # "basename" | "full"
show_windowed_waveform = False

# Partial tracking (after initial peak filtering):
partial_tracking = False
f0 = 263.0
partial_bandwidth_hz = 10.0
plot_partials = True

# Highlight colors (optional; keep None to use Plotly defaults):
spectrum_color = None
peaks_color = None
threshold_color = "Red"
partials_color = "rgba(0, 180, 0, 0.65)"
partials_band_fill = "rgba(0, 180, 0, 0.20)"

show_peaks = True
show_plot = True

plot_title_name = (
    audio_info["wav_source"] if plot_title_source == "source" else audio_info["wav_filename"]
)

peaks_df = analyze_signal(
    signal=cut_audio_data,
    sr=sample_rate,
    filename=plot_title_name,
    window_type=window_type,
    thresh_amp_low=thresh_amp_low,
    thresh_amp_high=thresh_amp_high,
    thresh_freq_low=thresh_freq_low,
    thresh_freq_high=thresh_freq_high,
    prominence=prominence,
    width=width,
    prominence_rel=prominence_rel,
    width_hz=width_hz,
    distance_hz=distance_hz,
    freq_axis_mode=freq_axis_mode,
    freq_axis_mix=freq_axis_mix,
    mixed_log_floor_hz=mixed_log_floor_hz,
    amp_axis_mode=amp_axis_mode,
    amp_axis_mix=amp_axis_mix,
    amp_log_floor=amp_log_floor,
    auto_plot_range=auto_plot_range,
    freq_plot_pad_hz=freq_plot_pad_hz,
    freq_plot_pad_frac=freq_plot_pad_frac,
    amp_plot_pad=amp_plot_pad,
    amp_plot_pad_frac=amp_plot_pad_frac,
    amp_plot_pad_ratio=amp_plot_pad_ratio,
    show_peaks=show_peaks,
    show_plot=show_plot,
    plot_width=plot_width,
    plot_height=plot_height,
    plot_title_name=plot_title_name,
    plot_title_mode=plot_title_mode,
    show_windowed_waveform=show_windowed_waveform,
    partial_tracking=partial_tracking,
    f0=f0,
    partial_bandwidth_hz=partial_bandwidth_hz,
    plot_partials=plot_partials,
    spectrum_color=spectrum_color,
    peaks_color=peaks_color,
    threshold_color=threshold_color,
    partials_color=partials_color,
    partials_band_fill=partials_band_fill,
)

## First Threshold Adjustment

We can already see that our initial threshold window was too broad. A log-scale amplitude view also helps a lot when detecting weak peaks.

In [ ]:
window_type = "boxcar"
thresh_amp_low = 0.0001
thresh_amp_high = 0.05 
thresh_freq_low = 20 
thresh_freq_high = 1000 

# Peak detection parameters
# Prefer the intuitive variants:
prominence_rel = 0.0001 # 0-1 range, try smaller values for more sensitive detection
width_hz = 0.1        # e.g. 5.0..30.0 to prefer broader peaks
distance_hz = 0.1     # e.g. 5.0..30.0 to avoid adjacent-bin peaks

# Keep these as None unless you want raw scipy behavior:
prominence = None
width = None

# Plot axis scaling:
freq_axis_mode = "linear"
freq_axis_mix = 0.5
mixed_log_floor_hz = 1.0

amp_axis_mode = "log"
amp_axis_mix = 0.5
amp_log_floor = 1e-12

# Auto-zoom plot ranges to your thresholds (+ padding):
auto_plot_range = True
freq_plot_pad_hz = None      # set e.g. 50 for +/-50 Hz padding
freq_plot_pad_frac = 0.05    # used when freq_plot_pad_hz is None
amp_plot_pad = None          # set e.g. 0.01 for +/-0.01 amplitude padding
amp_plot_pad_frac = 0.10     # used when amp_plot_pad is None
amp_plot_pad_ratio = 0.15    # only used when amp_axis_mode='log'

# Plot size / naming:
plot_width = 1200
plot_height = None
plot_title_source = "source"   # "filename" | "source"
plot_title_mode = "basename"     # "basename" | "full"
show_windowed_waveform = False

# Partial tracking (after initial peak filtering):
partial_tracking = False
f0 = 263.0
partial_bandwidth_hz = 10.0
plot_partials = True

# Highlight colors (optional; keep None to use Plotly defaults):
spectrum_color = None
peaks_color = None
threshold_color = "Red"
partials_color = "rgba(0, 180, 0, 0.65)"
partials_band_fill = "rgba(0, 180, 0, 0.20)"

show_peaks = True
show_plot = True

plot_title_name = (
    audio_info["wav_source"] if plot_title_source == "source" else audio_info["wav_filename"]
)

peaks_df = analyze_signal(
    signal=cut_audio_data,
    sr=sample_rate,
    filename=plot_title_name,
    window_type=window_type,
    thresh_amp_low=thresh_amp_low,
    thresh_amp_high=thresh_amp_high,
    thresh_freq_low=thresh_freq_low,
    thresh_freq_high=thresh_freq_high,
    prominence=prominence,
    width=width,
    prominence_rel=prominence_rel,
    width_hz=width_hz,
    distance_hz=distance_hz,
    freq_axis_mode=freq_axis_mode,
    freq_axis_mix=freq_axis_mix,
    mixed_log_floor_hz=mixed_log_floor_hz,
    amp_axis_mode=amp_axis_mode,
    amp_axis_mix=amp_axis_mix,
    amp_log_floor=amp_log_floor,
    auto_plot_range=auto_plot_range,
    freq_plot_pad_hz=freq_plot_pad_hz,
    freq_plot_pad_frac=freq_plot_pad_frac,
    amp_plot_pad=amp_plot_pad,
    amp_plot_pad_frac=amp_plot_pad_frac,
    amp_plot_pad_ratio=amp_plot_pad_ratio,
    show_peaks=show_peaks,
    show_plot=show_plot,
    plot_width=plot_width,
    plot_height=plot_height,
    plot_title_name=plot_title_name,
    plot_title_mode=plot_title_mode,
    show_windowed_waveform=show_windowed_waveform,
    partial_tracking=partial_tracking,
    f0=f0,
    partial_bandwidth_hz=partial_bandwidth_hz,
    plot_partials=plot_partials,
    spectrum_color=spectrum_color,
    peaks_color=peaks_color,
    threshold_color=threshold_color,
    partials_color=partials_color,
    partials_band_fill=partials_band_fill,
)

## Refined Peak Detection

Now we can see the underlying partial structure much more clearly. Using Plotly's pan/zoom view, we can define how far we want to lower the amplitude threshold to capture even weak partials. Let's keep `thresh_freq_high = 1000`, but lower `thresh_amp_low` to `0.00001` in linear FFT-magnitude units.

Let's also increase `width` and `distance` slightly to get more precise matches. A very small `distance` effectively treats nearly every FFT bin as a detected peak when `prominence` is low enough.

`prominence_rel = 0.00025` should be enough.

In [ ]:
window_type = "boxcar"
thresh_amp_low = 0.00001
thresh_amp_high = 0.05 
thresh_freq_low = 20 
thresh_freq_high = 1000 

# Prefer the intuitive variants:
prominence_rel = 0.00025 # 0-1 range, try smaller values for more sensitive detection
width_hz = 0.05        # e.g. 5.0..30.0 to prefer broader peaks
distance_hz = 1     # e.g. 5.0..30.0 to avoid adjacent-bin peaks

# Keep these as None unless you want raw scipy behavior:
prominence = None
width = None

# Plot axis scaling:
freq_axis_mode = "linear"
freq_axis_mix = 0.5
mixed_log_floor_hz = 1.0

amp_axis_mode = "log"
amp_axis_mix = 0.5
amp_log_floor = 1e-12

# Auto-zoom plot ranges to your thresholds (+ padding):
auto_plot_range = True
freq_plot_pad_hz = None      # set e.g. 50 for +/-50 Hz padding
freq_plot_pad_frac = 0.05    # used when freq_plot_pad_hz is None
amp_plot_pad = None          # set e.g. 0.01 for +/-0.01 amplitude padding
amp_plot_pad_frac = 0.10     # used when amp_plot_pad is None
amp_plot_pad_ratio = 0.15    # only used when amp_axis_mode='log'

# Plot size / naming:
plot_width = 1200
plot_height = None
plot_title_source = "source"   # "filename" | "source"
plot_title_mode = "basename"     # "basename" | "full"
show_windowed_waveform = False

# Partial tracking (after initial peak filtering):
partial_tracking = False
f0 = 263.0
partial_bandwidth_hz = 10.0
plot_partials = True

# Highlight colors (optional; keep None to use Plotly defaults):
spectrum_color = None
peaks_color = None
threshold_color = "Red"
partials_color = "rgba(0, 180, 0, 0.65)"
partials_band_fill = "rgba(0, 180, 0, 0.20)"

show_peaks = True
show_plot = True

plot_title_name = (
    audio_info["wav_source"] if plot_title_source == "source" else audio_info["wav_filename"]
)

peaks_df = analyze_signal(
    signal=cut_audio_data,
    sr=sample_rate,
    filename=plot_title_name,
    window_type=window_type,
    thresh_amp_low=thresh_amp_low,
    thresh_amp_high=thresh_amp_high,
    thresh_freq_low=thresh_freq_low,
    thresh_freq_high=thresh_freq_high,
    prominence=prominence,
    width=width,
    prominence_rel=prominence_rel,
    width_hz=width_hz,
    distance_hz=distance_hz,
    freq_axis_mode=freq_axis_mode,
    freq_axis_mix=freq_axis_mix,
    mixed_log_floor_hz=mixed_log_floor_hz,
    amp_axis_mode=amp_axis_mode,
    amp_axis_mix=amp_axis_mix,
    amp_log_floor=amp_log_floor,
    auto_plot_range=auto_plot_range,
    freq_plot_pad_hz=freq_plot_pad_hz,
    freq_plot_pad_frac=freq_plot_pad_frac,
    amp_plot_pad=amp_plot_pad,
    amp_plot_pad_frac=amp_plot_pad_frac,
    amp_plot_pad_ratio=amp_plot_pad_ratio,
    show_peaks=show_peaks,
    show_plot=show_plot,
    plot_width=plot_width,
    plot_height=plot_height,
    plot_title_name=plot_title_name,
    plot_title_mode=plot_title_mode,
    show_windowed_waveform=show_windowed_waveform,
    partial_tracking=partial_tracking,
    f0=f0,
    partial_bandwidth_hz=partial_bandwidth_hz,
    plot_partials=plot_partials,
    spectrum_color=spectrum_color,
    peaks_color=peaks_color,
    threshold_color=threshold_color,
    partials_color=partials_color,
    partials_band_fill=partials_band_fill,
)

## First Resynthesis

We see quite a lot of partials, particularly around our fundamental frequency (`f0 = 35.8 Hz`). How do they affect our perception of the sound?
Let's synthesize it.

In [ ]:
from audiospylt import expand_to_multi

df_multi = expand_to_multi(peaks_df, duration=5.0)

from audiospylt.plot_wave import estimate_sampling_frequency_and_time_vector

sampling_frequency_updated, delta_t_updated, duration = estimate_sampling_frequency_and_time_vector(df_multi)

print(f"Estimated Optimal Sampling Frequency: {sampling_frequency_updated:.3f} Hz")
print(f"Sampling Interval (Delta t): {delta_t_updated:.6f} seconds")
print(f"Duration (highest value from time_stop column): {duration:.3f} seconds")


In [ ]:
from audiospylt.plot_wave import plot_waves

# Define the time step k (seconds) and corresponding sample rate
k=0.0005
sample_rate_resynth = 1.0 / k

# Optional FFT/spectrogram settings (forwarded to `audiospylt.audio_utils.plot_spectrogram`)
spectrogram_kwargs = {
    # Y-axis scaling
    "y_axis_mode": "log",           # 'linear' | 'log' | 'mel' | 'mixed'
    "y_axis_mix": 0.5,               # 0..1 (only for 'mixed': blends linear<->log spacing)
    "mixed_log_floor_hz": 1.0,       # >0 (only for 'mixed': log floor; keep >0 to avoid log(0))

    # FFT / time-frequency settings
    "n_fft": 256,                    # window/FFT size; larger -> finer frequency bins, coarser time bins
    "window_type": "hann",          # window shape (scipy.signal.get_window)
    "overlap": 0.75,                 # fraction of window overlap; keep <0.95 (hop must stay >0)
    "oversample_factor": 1.0,        # >=1.0; zero-pads the FFT (more interpolated frequency grid, same time grid)

    # Mel settings (only for y_axis_mode='mel')
    "mel_bins": 128,
    "mel_fmax": sample_rate_resynth / 2,     # cap at Nyquist

    # Rendering / framing (affects both computation and how the plot aligns in time)
    "scaling": "density",           # SciPy scaling: 'density' (PSD-like, per Hz) | 'spectrum' (per-bin)
    "mode": "magnitude",            # 'magnitude' = |STFT|, 'psd' = power spectral density (scaling applies)
    "cmap": "Magma",                # Plotly continuous colorscale name (e.g. 'Viridis', 'Magma', ...)

    # STFT edge handling:
    # - None: no padding ("valid" framing); first/last ~n_fft/2 may be missing in the visible extent
    # - 'zeros': pad both sides by n_fft//2 so a frame can be centered at t=0 (shows pre-zero padding)
    # - 'zeros_end': pad only the end so the first full window starts at t=0 (often cleaner for onsets)
    # - any other string: forwarded as `numpy.pad` mode (pads both sides by n_fft//2)
    "boundary": "zeros",

    # If True, add minimal end-padding so the last hop boundary inside the *original* signal
    # still gets a frame (helps avoid an overlap-dependent "gap" near the end).
    "padded": True,

    # X-axis view range:
    # - 'signal': force x-axis to [0, duration] (aligns with waveform)
    # - 'stft': show the full extent of generated frames (can include padding tails)
    "time_range": "signal",

    "show": True,                    # show the Plotly figure now (`plot_waves` defaults to `True` if omitted)
}

# Default behavior: show the synthesized waveform only (time-step `k` resolution intact).
# Set show_fft=True when you want the spectrogram.
y_combined = plot_waves(
    df_multi,
    k,
    # Convenience fade duration (seconds) applied to *both* ends of each event.
    # - 0 disables per-event edge fades (useful when you want exact hard boundaries)
    # - You can override per-row with df columns 'fade_in_s' / 'fade_out_s'
    # - Or pass fade_in_s=... / fade_out_s=... to control globally (takes precedence over edge_fade_s)
    edge_fade_s=0.05,
    # Phase handling inside each event:
    # - 'reset': phase restarts at each event start (simple; most prone to boundary clicks)
    # - 'global': absolute-time-referenced phase; keeps phase continuous across adjacent events
    #             when freq_start==freq_stop (constant-frequency bins)
    # - 'chirp': integrates a linear ramp so instantaneous frequency follows
    #             freq_start->freq_stop *within* the event (does not guarantee continuity across events)
    phase_mode="chirp",
    show_waveform=True,
    show_fft=True,
    spectrogram_kwargs=spectrogram_kwargs,
    plot_width=plot_width
)

In [ ]:
from audiospylt.generate_wave_file import render_audio

player = True
save_audio = False

fs_initial = 1 / k  # The initial sampling rate is the inverse of the time step (k)

render_audio(
    y_combined,
    fs_initial,
    fs_target_name="44.1kHz",

    # If True, downsampling to fs_target_name will *not* low-pass above Nyquist.
    # Instead it intentionally allows aliasing ("Nyquist mirroring") so partials above
    # fs_target/2 fold back into-band while the exported file stays at a common sample rate.
    alias_above_nyquist=False,

    bit_rate=24,
    filename_template="testing_{fs_target_name}_{bit_rate}bit_{timestamp}",
    timestamp_format="%Y-%m-%d_%H-%M-%S",
    save_audio=save_audio,
    player=player,
    sanitize=True,
    verbose=True,
);


## First Comparison

How close does it sound to the original?

Here is the original reference again:

In [ ]:
from audiospylt import render_audio

# Assume these already exist:
# cut_audio_data: 1D NumPy array with your cut region
# sample_rate:    int or float, Hz (same rate as cut_audio_data)

_ = render_audio(
    cut_audio_data,
    sample_rate,           # fs_initial
    fs_target_name="source",  # keep the same sample rate (no resampling)
    bit_rate=24,
    save_audio=False,         # don't write a file, just return (audio, fs)
    player=True,              # show an IPython Audio player
    sanitize=True,
    verbose=True,
)

## Listening for the Difference

Can we hear the difference?

How much does the high-frequency noise or hiss in the original affect the sonic perception of the sound? We are obviously not resynthesizing those higher-frequency components here, because our peak analysis is limited to a `1000 Hz` threshold window and we therefore use a `2 kHz` sample rate for rendering.
Does the noise affect the perception of harmonic structure?

What about the fifth Smalley mentioned?
Let's take another look at the spectrum.

In [ ]:
from IPython.display import Image, display

display(Image(filename="img/spectrum_parm.png"))

## The Fifth and Partial Tracking

If we consider our lowest peak (`35.8 Hz`) as the fundamental frequency, we can see that the strong peak that appears to be the fifth (the third partial) is actually slightly low: about `106 Hz` instead of the expected `107.2 Hz`. The deviation from the mathematically exact `3 * 35.8 = 107.4 Hz` is caused by our `0.2 Hz` frequency resolution, which is already very precise given the full `5 s` sample length (`110251` samples).
This should create a low-frequency modulation that is barely, if at all, recognizable in the spectrogram, but it is clearly visible in the waveform of both the original and the synthesized sound.

Let's try to track the partials with audiospylt's partial filter using a `5 Hz` bandwidth (including all peaks within `+-5 Hz` of each partial) for both `35.8 Hz` and `106 Hz` as candidate `f0` values.

In [ ]:
window_type = "boxcar"
thresh_amp_low = 0.00001
thresh_amp_high = 0.05 
thresh_freq_low = 20 
thresh_freq_high = 1000 

# Prefer the intuitive variants:
prominence_rel = 0.00025 # 0-1 range, try smaller values for more sensitive detection
width_hz = 0.05        # e.g. 5.0..30.0 to prefer broader peaks
distance_hz = 1     # e.g. 5.0..30.0 to avoid adjacent-bin peaks

# Keep these as None unless you want raw scipy behavior:
prominence = None
width = None

# Plot axis scaling:
freq_axis_mode = "linear"
freq_axis_mix = 0.5
mixed_log_floor_hz = 1.0

amp_axis_mode = "log"
amp_axis_mix = 0.5
amp_log_floor = 1e-12

# Auto-zoom plot ranges to your thresholds (+ padding):
auto_plot_range = True
freq_plot_pad_hz = None      # set e.g. 50 for +/-50 Hz padding
freq_plot_pad_frac = 0.05    # used when freq_plot_pad_hz is None
amp_plot_pad = None          # set e.g. 0.01 for +/-0.01 amplitude padding
amp_plot_pad_frac = 0.10     # used when amp_plot_pad is None
amp_plot_pad_ratio = 0.15    # only used when amp_axis_mode='log'

# Plot size / naming:
plot_width = 1200
plot_height = None
plot_title_source = "source"   # "filename" | "source"
plot_title_mode = "basename"     # "basename" | "full"
show_windowed_waveform = False

# Partial tracking (after initial peak filtering):
partial_tracking = True
#f0 = 35.8
f0 = [35.8, 106.0]
partial_bandwidth_hz = 5.0
plot_partials = True

# Highlight colors (optional; keep None to use Plotly defaults):
spectrum_color = None
peaks_color = None
threshold_color = "Red"
partials_color = "rgba(0, 180, 0, 0.65)"
partials_band_fill = "rgba(0, 180, 0, 0.20)"

show_peaks = True
show_plot = True

plot_title_name = (
    audio_info["wav_source"] if plot_title_source == "source" else audio_info["wav_filename"]
)

peaks_df_partials = analyze_signal(
    signal=cut_audio_data,
    sr=sample_rate,
    filename=plot_title_name,
    window_type=window_type,
    thresh_amp_low=thresh_amp_low,
    thresh_amp_high=thresh_amp_high,
    thresh_freq_low=thresh_freq_low,
    thresh_freq_high=thresh_freq_high,
    prominence=prominence,
    width=width,
    prominence_rel=prominence_rel,
    width_hz=width_hz,
    distance_hz=distance_hz,
    freq_axis_mode=freq_axis_mode,
    freq_axis_mix=freq_axis_mix,
    mixed_log_floor_hz=mixed_log_floor_hz,
    amp_axis_mode=amp_axis_mode,
    amp_axis_mix=amp_axis_mix,
    amp_log_floor=amp_log_floor,
    auto_plot_range=auto_plot_range,
    freq_plot_pad_hz=freq_plot_pad_hz,
    freq_plot_pad_frac=freq_plot_pad_frac,
    amp_plot_pad=amp_plot_pad,
    amp_plot_pad_frac=amp_plot_pad_frac,
    amp_plot_pad_ratio=amp_plot_pad_ratio,
    show_peaks=show_peaks,
    show_plot=show_plot,
    plot_width=plot_width,
    plot_height=plot_height,
    plot_title_name=plot_title_name,
    plot_title_mode=plot_title_mode,
    show_windowed_waveform=show_windowed_waveform,
    partial_tracking=partial_tracking,
    f0=f0,
    partial_bandwidth_hz=partial_bandwidth_hz,
    plot_partials=plot_partials,
    spectrum_color=spectrum_color,
    peaks_color=peaks_color,
    threshold_color=threshold_color,
    partials_color=partials_color,
    partials_band_fill=partials_band_fill,
)

## Residual Peaks and `search_f0(...)`

As we can see, we cover almost all prominent peaks. Only a few outliers, for example at `500.4`, `565`, `706.2`, `918.2`, and `988.8 Hz`, remain outside. But are they part of another partial series? Let's use `search_f0(...)` to find out.

In [ ]:
from audiospylt import search_f0, summarize_result

freqs = [500.4, 565.0, 706.2, 918.2, 988.8]
result = search_f0(freqs, include_plot=True, show_plot=True)

summarize_result(
    "Baseline list input",
    result,
    top_n=5,
    precision=3,
    suppress=True,
    floatmode="fixed",
)


## Multi-`f0` Reconstruction

Now we can use all three `f0` values to reconstruct the spectrum based on the estimated fundamentals.

In [ ]:
window_type = "boxcar"
thresh_amp_low = 0.00001
thresh_amp_high = 0.05 
thresh_freq_low = 20 
thresh_freq_high = 1000 

# Prefer the intuitive variants:
prominence_rel = 0.00025 # 0-1 range, try smaller values for more sensitive detection
width_hz = 0.05        # e.g. 5.0..30.0 to prefer broader peaks
distance_hz = 1     # e.g. 5.0..30.0 to avoid adjacent-bin peaks

# Keep these as None unless you want raw scipy behavior:
prominence = None
width = None

# Plot axis scaling:
freq_axis_mode = "linear"
freq_axis_mix = 0.5
mixed_log_floor_hz = 1.0

amp_axis_mode = "log"
amp_axis_mix = 0.5
amp_log_floor = 1e-12

# Auto-zoom plot ranges to your thresholds (+ padding):
auto_plot_range = True
freq_plot_pad_hz = None      # set e.g. 50 for +/-50 Hz padding
freq_plot_pad_frac = 0.05    # used when freq_plot_pad_hz is None
amp_plot_pad = None          # set e.g. 0.01 for +/-0.01 amplitude padding
amp_plot_pad_frac = 0.10     # used when amp_plot_pad is None
amp_plot_pad_ratio = 0.15    # only used when amp_axis_mode='log'

# Plot size / naming:
plot_width = 1200
plot_height = None
plot_title_source = "source"   # "filename" | "source"
plot_title_mode = "basename"     # "basename" | "full"
show_windowed_waveform = False

# Partial tracking (after initial peak filtering):
partial_tracking = True
#f0 = 35.8
f0 = [35.8, 70.6, 106.0]
partial_bandwidth_hz = 5.0
plot_partials = True

# Highlight colors (optional; keep None to use Plotly defaults):
spectrum_color = None
peaks_color = None
threshold_color = "Red"
partials_color = "rgba(0, 180, 0, 0.65)"
partials_band_fill = "rgba(0, 180, 0, 0.20)"

show_peaks = True
show_plot = True

plot_title_name = (
    audio_info["wav_source"] if plot_title_source == "source" else audio_info["wav_filename"]
)

peaks_df_partials = analyze_signal(
    signal=cut_audio_data,
    sr=sample_rate,
    filename=plot_title_name,
    window_type=window_type,
    thresh_amp_low=thresh_amp_low,
    thresh_amp_high=thresh_amp_high,
    thresh_freq_low=thresh_freq_low,
    thresh_freq_high=thresh_freq_high,
    prominence=prominence,
    width=width,
    prominence_rel=prominence_rel,
    width_hz=width_hz,
    distance_hz=distance_hz,
    freq_axis_mode=freq_axis_mode,
    freq_axis_mix=freq_axis_mix,
    mixed_log_floor_hz=mixed_log_floor_hz,
    amp_axis_mode=amp_axis_mode,
    amp_axis_mix=amp_axis_mix,
    amp_log_floor=amp_log_floor,
    auto_plot_range=auto_plot_range,
    freq_plot_pad_hz=freq_plot_pad_hz,
    freq_plot_pad_frac=freq_plot_pad_frac,
    amp_plot_pad=amp_plot_pad,
    amp_plot_pad_frac=amp_plot_pad_frac,
    amp_plot_pad_ratio=amp_plot_pad_ratio,
    show_peaks=show_peaks,
    show_plot=show_plot,
    plot_width=plot_width,
    plot_height=plot_height,
    plot_title_name=plot_title_name,
    plot_title_mode=plot_title_mode,
    show_windowed_waveform=show_windowed_waveform,
    partial_tracking=partial_tracking,
    f0=f0,
    partial_bandwidth_hz=partial_bandwidth_hz,
    plot_partials=plot_partials,
    spectrum_color=spectrum_color,
    peaks_color=peaks_color,
    threshold_color=threshold_color,
    partials_color=partials_color,
    partials_band_fill=partials_band_fill,
)

In [ ]:
df_multi_f0 = expand_to_multi(peaks_df_partials, duration=5.0)

from audiospylt.plot_wave import estimate_sampling_frequency_and_time_vector

sampling_frequency_updated, delta_t_updated, duration = estimate_sampling_frequency_and_time_vector(df_multi)

print(f"Estimated Optimal Sampling Frequency: {sampling_frequency_updated:.3f} Hz")
print(f"Sampling Interval (Delta t): {delta_t_updated:.6f} seconds")
print(f"Duration (highest value from time_stop column): {duration:.3f} seconds")

In [ ]:
from audiospylt.plot_wave import plot_waves

# Define the time step k (seconds) and corresponding sample rate
k=0.0005
sample_rate_resynth = 1.0 / k

# Optional FFT/spectrogram settings (forwarded to `audiospylt.audio_utils.plot_spectrogram`)
spectrogram_kwargs = {
    # Y-axis scaling
    "y_axis_mode": "log",           # 'linear' | 'log' | 'mel' | 'mixed'
    "y_axis_mix": 0.5,               # 0..1 (only for 'mixed': blends linear<->log spacing)
    "mixed_log_floor_hz": 1.0,       # >0 (only for 'mixed': log floor; keep >0 to avoid log(0))

    # FFT / time-frequency settings
    "n_fft": 256,                    # window/FFT size; larger -> finer frequency bins, coarser time bins
    "window_type": "hann",          # window shape (scipy.signal.get_window)
    "overlap": 0.75,                 # fraction of window overlap; keep <0.95 (hop must stay >0)
    "oversample_factor": 1.0,        # >=1.0; zero-pads the FFT (more interpolated frequency grid, same time grid)

    # Mel settings (only for y_axis_mode='mel')
    "mel_bins": 128,
    "mel_fmax": sample_rate_resynth / 2,     # cap at Nyquist

    # Rendering / framing (affects both computation and how the plot aligns in time)
    "scaling": "density",           # SciPy scaling: 'density' (PSD-like, per Hz) | 'spectrum' (per-bin)
    "mode": "magnitude",            # 'magnitude' = |STFT|, 'psd' = power spectral density (scaling applies)
    "cmap": "Magma",                # Plotly continuous colorscale name (e.g. 'Viridis', 'Magma', ...)

    # STFT edge handling:
    # - None: no padding ("valid" framing); first/last ~n_fft/2 may be missing in the visible extent
    # - 'zeros': pad both sides by n_fft//2 so a frame can be centered at t=0 (shows pre-zero padding)
    # - 'zeros_end': pad only the end so the first full window starts at t=0 (often cleaner for onsets)
    # - any other string: forwarded as `numpy.pad` mode (pads both sides by n_fft//2)
    "boundary": "zeros",

    # If True, add minimal end-padding so the last hop boundary inside the *original* signal
    # still gets a frame (helps avoid an overlap-dependent "gap" near the end).
    "padded": True,

    # X-axis view range:
    # - 'signal': force x-axis to [0, duration] (aligns with waveform)
    # - 'stft': show the full extent of generated frames (can include padding tails)
    "time_range": "signal",

    "show": True,                    # show the Plotly figure now (`plot_waves` defaults to `True` if omitted)
}

# Default behavior: show the synthesized waveform only (time-step `k` resolution intact).
# Set show_fft=True when you want the spectrogram.
y_combined_f0 = plot_waves(
    df_multi_f0,
    k,
    # Convenience fade duration (seconds) applied to *both* ends of each event.
    # - 0 disables per-event edge fades (useful when you want exact hard boundaries)
    # - You can override per-row with df columns 'fade_in_s' / 'fade_out_s'
    # - Or pass fade_in_s=... / fade_out_s=... to control globally (takes precedence over edge_fade_s)
    edge_fade_s=0.05,
    # Phase handling inside each event:
    # - 'reset': phase restarts at each event start (simple; most prone to boundary clicks)
    # - 'global': absolute-time-referenced phase; keeps phase continuous across adjacent events
    #             when freq_start==freq_stop (constant-frequency bins)
    # - 'chirp': integrates a linear ramp so instantaneous frequency follows
    #             freq_start->freq_stop *within* the event (does not guarantee continuity across events)
    phase_mode="chirp",
    show_waveform=True,
    show_fft=True,
    spectrogram_kwargs=spectrogram_kwargs,
    plot_width=plot_width
)

In [ ]:
from audiospylt.generate_wave_file import render_audio

player = True
save_audio = False

fs_initial = 1 / k  # The initial sampling rate is the inverse of the time step (k)

render_audio(
    y_combined_f0,
    fs_initial,
    fs_target_name="44.1kHz",

    # If True, downsampling to fs_target_name will *not* low-pass above Nyquist.
    # Instead it intentionally allows aliasing ("Nyquist mirroring") so partials above
    # fs_target/2 fold back into-band while the exported file stays at a common sample rate.
    alias_above_nyquist=False,

    bit_rate=24,
    filename_template="testing_{fs_target_name}_{bit_rate}bit_{timestamp}",
    timestamp_format="%Y-%m-%d_%H-%M-%S",
    save_audio=save_audio,
    player=player,
    sanitize=True,
    verbose=True,
);


## Final A/B Comparison

Here are the original and the resyntheses for a back-to-back comparison:

In [ ]:
for label, y, sr in [
    ("original", cut_audio_data, sample_rate),
    ("resynth", y_combined, fs_initial),
    ("resynth f0", y_combined_f0, fs_initial),
]:
    print(label)
    _ = render_audio(
        y,
        sr,
        fs_target_name="44.1kHz",   # or "48kHz", "source", etc.
        save_audio=False,
        player=True,
        verbose=False,
    )



## Symbolic Notation (MEI)

If we want to visualize the peaks as symbolic notation, we can use audiospylt's MEI-based processing. The sequence view with `half_tone_deviation` resolution is particularly useful because it shows the microtonal deviations in MIDI cents.

In [ ]:
from audiospylt.mei import process_and_visualize

import os
if not os.path.exists('mei'):
    os.makedirs('mei')
    print("Created 'mei' directory for saved files.")

process_and_visualize(
    peaks_df_partials,
    resolution='half_tone_deviation',
    clef_config='GF',
    note_order='original',
    display_cent_deviation=True,
    include_natural_accidentals=True,
    display_mode='sequence',
    display_df=True,
    save_mei=False,
    mei_save_path='mei/parm_resynth_sequence.mei', # Example path
    save_svg=False,
    notebook_white_background=True,
)
